# Análise de Projetos do FNMA (1990 - 2024)

**Fundo Nacional do Meio Ambiente** - análise exploratória dos projetos ambientais financiados pelo governo federal entre 1990 e 2024.

## Objetivo

Entender como os recursos do FNMA foram distribuídos ao longo de mais de três décadas: quais temas, regiões e biomas concentraram mais investimento, como isso evoluiu no tempo, e que padrões (ou problemas) os dados revelam sobre a política de financiamento ambiental no Brasil.

## Perguntas que guiam a análise

1. Quais temas e biomas concentram mais projetos e mais recursos?
2. Como o financiamento se distribui entre as regiões do país?
3. Como o volume de projetos e recursos evoluiu ao longo do tempo?
4. Os dados têm problemas de qualidade que precisam ser tratados antes de qualquer conclusão?

## Fonte dos dados

Dados abertos do [Ministério do Meio Ambiente](https://www.gov.br/mma/pt-br). Arquivo `fnma_1990_2024.csv`, 1.495 projetos, 19 colunas (tema, região, bioma, valores financeiros, datas, instituição executora, entre outras).


In [51]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
px.defaults.template = 'plotly_white'


## 1. Carregamento e inspeção inicial

In [52]:
df = pd.read_csv('../data/fnma_1990_2024.csv', sep=None, engine='python')

# a coluna 'Ano' vem com um caractere BOM invisível (\ufeff) por causa da codificação do arquivo
df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]

print(f"Linhas: {len(df):,}".replace(',', '.'))
print(f"Colunas: {len(df.columns)}")
df.head()


Linhas: 1.495
Colunas: 19


,Ano,Nº do Instrumento de Repasse,Nº Interno,Tema,Instituição Executora,Título do Projeto,UF,Região Geográfica,Cidade da Instituição Executora,Bioma,Esfera Institucional,Data Assinatura,Data de Publicação no DOU,Data de Fim da Vigência,Recursos do FNMA (R$),Recursos de CP (R$),Valor Total do Projeto (R$),Tipo de Seleção do Projeto,Edital ou Termo de Referência de Origem do Projeto
0,1990,NaN,CV001/90,Educação Ambiental,Fundação Nacional do Índio - Funai,I Encontro sobre Meio Ambiente Indígena,DF,Centro-Oeste,NaN,Cerrado,Federal,07/12/1990,NaN,31/12/1990,"2.837,01",0,"2.837,01",Demanda Espontânea (DE),NaN
1,1990,NaN,CV002/90,Gestão de Áreas Protegidas,Núcleo de Cultura Indígena da União das Nações...,Serra do Roncador/Xavante: Manejo Indígena de ...,GO,Centro-Oeste,NaN,Cerrado,OSC,14/11/1990,NaN,31/12/1990,"41.212,07",0,"41.212,07",Demanda Espontânea (DE),NaN
2,1990,NaN,CV003/90,Conservação e Manejo da Biodiversidade,Fundação Pró-natureza – Funatura,Conservação da Avifauna da Amazônia,DF,Centro-Oeste,NaN,Amazônia,OSC,13/11/1990,NaN,31/12/1990,"22.907,20",0,"22.907,20",Demanda Espontânea (DE),NaN
3,1990,NaN,CV004/90,Gestão de Áreas Protegidas,Núcleo de Cultura Indígena da União das Nações...,Metareilá/Suruí: Manejo Indígena de Floresta T...,GO,Centro-Oeste,NaN,Amazônia,OSC,14/11/1990,NaN,31/12/1990,"88.484,75",0,"88.484,75",Demanda Espontânea (DE),NaN
4,1990,NaN,CV005/90,Amazônia Sustentável,Prefeitura Municipal de Curralinho,Extensão Florestal e Pesquisa nas Áreas das Fl...,AM,Norte,NaN,Amazônia,Municipal,13/11/1990,22/11/1990,31/12/1990,"304.654,39",0,"304.654,39",Demanda Espontânea (DE),NaN


In [53]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1495 entries, 0 to 1494
Data columns (total 19 columns):
 #   Column                                              Non-Null Count  Dtype
---  ------                                              --------------  -----
 0   Ano                                                 1495 non-null   int64
 1   Nº do Instrumento de Repasse                        97 non-null     str  
 2   Nº  Interno                                         1484 non-null   str  
 3   Tema                                                1495 non-null   str  
 4   Instituição Executora                               1495 non-null   str  
 5   Título do Projeto                                   1495 non-null   str  
 6   UF                                                  1495 non-null   str  
 7   Região Geográfica                                   1495 non-null   str  
 8   Cidade da Instituição Executora                     648 non-null    str  
 9   Bioma                         

## 2. Qualidade dos dados

Antes de qualquer análise, é preciso checar o que os dados brutos realmente contêm. Aqui encontrei pelo menos três problemas clássicos: **inconsistência de texto** (espaços extras criando categorias duplicadas), **valores monetários como texto** no formato brasileiro (`1.234,56`), e **datas como string**.


In [54]:
# valores ausentes por coluna
df.isna().sum().sort_values(ascending=False)


Nº do Instrumento de Repasse                          1398
Edital ou Termo de Referência de Origem do Projeto     964
Cidade da Instituição Executora                        847
Data Assinatura                                        412
Data de Fim da Vigência                                 35
Data de Publicação no DOU                               28
Nº  Interno                                             11
Bioma                                                   10
Ano                                                      0
Tema                                                     0
Instituição Executora                                    0
Esfera Institucional                                     0
Região Geográfica                                        0
Título do Projeto                                        0
UF                                                       0
Recursos do FNMA (R$)                                    0
Recursos de CP (R$)                                     

Colunas como `Nº do Instrumento de Repasse`, `Cidade da Instituição Executora` e `Edital ou Termo de Referência` têm muitos nulos, mas não são centrais para as perguntas da análise, então optei poe não tratar agora. `Data Assinatura` tem 412 valores ausentes, o que limita (mas não invalida) qualquer análise de duração de projeto mais adiante.

**Duplicatas:**


In [55]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")


Linhas duplicadas: 0


**Inconsistência de texto:** colunas categóricas como `Tema`, `Região Geográfica` e `Bioma` têm espaços extras que fazem o pandas tratar `'Sudeste'` e `'Sudeste '` como duas categorias diferentes.


In [56]:
print("Tema - categorias antes da limpeza:", df['Tema'].nunique())
print("Região Geográfica antes:", sorted(df['Região Geográfica'].unique()))


Tema - categorias antes da limpeza: 16
Região Geográfica antes: ['Centro-Oeste', 'Nordeste', 'Nordeste  ', 'Norte', 'Norte ', 'Sudeste', 'Sudeste ', 'Sul', 'Sul ']


In [57]:
colunas_categoricas = ['Tema', 'UF', 'Região Geográfica', 'Bioma',
                        'Esfera Institucional', 'Tipo de Seleção do Projeto']

for coluna in colunas_categoricas:
    df[coluna] = df[coluna].str.strip()

print("Tema - categorias depois da limpeza:", df['Tema'].nunique())
print("Região Geográfica depois:", sorted(df['Região Geográfica'].unique()))


Tema - categorias depois da limpeza: 13
Região Geográfica depois: ['Centro-Oeste', 'Nordeste', 'Norte', 'Sudeste', 'Sul']


A limpeza reduziu `Tema` de 16 para 13 categorias reais, três eram duplicatas por espaço extra. Sem esse passo, qualquer contagem por tema estaria fragmentada e sub-representando os temas afetados.

**Valores monetários:** as três colunas de valor (`Recursos do FNMA`, `Recursos de CP`, `Valor Total do Projeto`) vêm como texto no formato brasileiro (`R$ 41.212,07`), o que impede qualquer soma ou média diretamente.


In [61]:
def valor_brl_para_float(valor):
    """Converte string monetária no formato brasileiro ('41.212,07') para float."""
    if pd.isna(valor):
        return None
    if isinstance(valor, (int, float)):
        return float(valor)
    return float(str(valor).replace('.', '').replace(',', '.'))

colunas_valor = ['Recursos do FNMA (R$)', 'Recursos de CP (R$)', 'Valor Total do Projeto (R$)']

for coluna in colunas_valor:
    df[coluna] = df[coluna].apply(valor_brl_para_float)

resumo = df[colunas_valor].describe()
resumo.style.format('{:,.2f}').format('{:,.0f}', subset=pd.IndexSlice['count', :])

,Recursos do FNMA (R$),Recursos de CP (R$),Valor Total do Projeto (R$)
count,"1,495","1,495","1,495"
mean,"209,872.94","65,379.40","275,389.67"
std,"418,197.35","126,031.04","456,566.67"
min,"1,515.45",0.00,"2,020.45"
25%,"48,050.00","9,688.34","67,597.50"
50%,"124,000.00","28,992.00","175,970.00"
75%,"265,737.99","75,000.00","353,334.50"
max,"12,040,350.64","3,251,200.00","12,052,405.00"


**Datas:** convertidas de string (`dd/mm/aaaa`) para `datetime`, o que permite calcular a duração de cada projeto (assinatura → fim da vigência).


In [64]:
colunas_data = ['Data Assinatura', 'Data de Publicação no DOU', 'Data de Fim da Vigência']

for coluna in colunas_data:
    df[coluna] = pd.to_datetime(df[coluna], format='%d/%m/%Y', errors='coerce')

df['Duração (dias)'] = (df['Data de Fim da Vigência'] - df['Data Assinatura']).dt.days
df['Duração (dias)'].describe()


count   1,051.00
mean      504.63
std       347.75
min      -233.00
25%       283.50
50%       426.00
75%       717.00
max     2,406.00
Name: Duração (dias), dtype: float64

Duas linhas têm duração **negativa**, a data de fim da vigência é anterior à data de assinatura, o que é logicamente impossível e indica erro de digitação na base original. Fica registrado aqui como limitação dos dados, e essas linhas são excluídas apenas da análise de duração (não do restante).


In [65]:
projetos_com_erro_de_data = df[df['Duração (dias)'] < 0]
print(f"Projetos com duração negativa (erro de digitação): {len(projetos_com_erro_de_data)}")
projetos_com_erro_de_data[['Título do Projeto', 'Data Assinatura', 'Data de Fim da Vigência']]


Projetos com duração negativa (erro de digitação): 2


,Título do Projeto,Data Assinatura,Data de Fim da Vigência
980,"Manejo de Pesca, Maricultura e Turismo Respons...",2003-07-03,2003-03-31
1470,Criação de um Centro de Educação e Cooperação ...,2023-12-19,2023-04-30


## 3. Panorama geral: onde foram os projetos e os recursos

Para evitar repetir a mesma lógica de `groupby` + gráfico de barras em cada corte (ano, tema, região...), uma função reutilizável concentra essa lógica.

In [66]:
def contagem_por_categoria(df, coluna, top_n=None, ordenar_por='Projetos'):
    """Conta projetos por categoria e retorna um DataFrame pronto para plotar."""
    resultado = df[coluna].value_counts().reset_index()
    resultado.columns = [coluna, 'Projetos']
    resultado = resultado.sort_values('Projetos', ascending=True)
    if top_n:
        resultado = resultado.tail(top_n)
    return resultado


def grafico_barra_horizontal(dados, coluna, titulo, escala_cor='Teal'):
    return px.bar(
        dados, x='Projetos', y=coluna, orientation='h',
        title=titulo, color='Projetos', color_continuous_scale=escala_cor,
        text='Projetos',
    ).update_traces(textposition='outside').update_layout(showlegend=False)


In [67]:
por_ano = df.groupby('Ano').size().reset_index(name='Projetos')

fig = px.bar(
    por_ano, x='Ano', y='Projetos', title='Projetos por ano',
    color='Projetos', color_continuous_scale='Blues',
)
fig.update_layout(xaxis=dict(dtick=2))
fig.show()


Há um pico claro de atividade entre **1993 e 2006**, seguido de um hiato quase total entre 2007 e 2023, e uma retomada em 2024. Isso provavelmente reflete mudanças na política de financiamento ambiental federal ao longo do tempo, mais do que uma queda gradual de demanda. Uma hipótese, não uma conclusão fechada, já que os dados por si só não explicam a causa.


In [68]:
por_tema = contagem_por_categoria(df, 'Tema')
grafico_barra_horizontal(por_tema, 'Tema', 'Projetos por tema').show()


In [76]:
por_regiao = contagem_por_categoria(df, 'Região Geográfica')
grafico_barra_horizontal(por_regiao, 'Região Geográfica', 'Projetos por região', escala_cor='Purples').show()


**Biomas** merecem tratamento especial: várias linhas têm mais de um bioma na mesma célula (ex.: `"Mata Atlântica / Cerrado"`), porque o projeto atravessa mais de um bioma. Contar essas células como categorias únicas sub-representaria os biomas individuais.


In [77]:
biomas_normalizados = df['Bioma'].dropna().str.replace(' e ', ' / ').str.split('/')
biomas_explodidos = biomas_normalizados.explode().str.strip()

por_bioma = biomas_explodidos.value_counts().reset_index()
por_bioma.columns = ['Bioma', 'Projetos']
por_bioma = por_bioma.sort_values('Projetos', ascending=True)

grafico_barra_horizontal(por_bioma, 'Bioma', 'Projetos por bioma (contagem por bioma individual)', escala_cor='Greens').show()


**Combinação vs. bioma individual:** projetos com mais de um bioma na mesma célula (ex.: `"Mata Atlântica / Cerrado"`) foram contados uma vez para cada bioma que tocam, por isso a soma das contagens acima (1.494) é maior que o número de projetos com bioma preenchido (1.485; 10 projetos não têm bioma informado). A tabela abaixo mostra as duas visões lado a lado: quantos projetos têm exatamente aquela categoria (sem separar combinações) vs. quantos *tocam* aquele bioma (incluindo combinações).

In [78]:
biomas_puros = df['Bioma'].value_counts().reset_index()
biomas_puros.columns = ['Bioma', 'Projetos (categoria exata, sem separar combinações)']

biomas_qualquer = por_bioma.rename(columns={'Projetos': 'Projetos (contando cada bioma tocado)'})

comparacao_biomas = biomas_puros.merge(biomas_qualquer, on='Bioma', how='outer').fillna(0)
comparacao_biomas = comparacao_biomas.sort_values('Projetos (contando cada bioma tocado)', ascending=False)
comparacao_biomas

,Bioma,"Projetos (categoria exata, sem separar combinações)",Projetos (contando cada bioma tocado)
6,Mata Atlântica,751,757.00
0,Amazônia,306,307.00
2,Cerrado,209,216.00
1,Caatinga,137,140.00
5,Marítimo,46,46.00
11,Pampa,14,15.00
12,Pantanal,11,11.00
10,Orla,2,2.00
3,Cerrado / Amazônia,1,0.00
4,Cerrado / Caatinga,2,0.00


**Recursos financeiros:** a distribuição de valor por projeto é bastante assimétrica, a mediana (R$ 176 mil) é bem menor que a média (R$ 275 mil), sinal de que poucos projetos de grande porte puxam a média para cima. Um histograma em escala logarítmica mostra isso melhor do que uma escala linear.


In [79]:
fig = px.histogram(
    df, x='Valor Total do Projeto (R$)', nbins=50, log_y=True,
    title='Distribuição do valor total dos projetos (eixo Y em escala log)',
)
fig.show()


## 4. Cruzando dimensões: tema x região e evolução no tempo

Contagens isoladas por tema ou por região escondem um padrão importante: será que os temas se distribuem igualmente entre as regiões, ou cada região tem sua própria "vocação" temática? Um heatmap responde isso de forma direta.


In [81]:
tabela_cruzada = pd.crosstab(df['Tema'], df['Região Geográfica'])

fig = px.imshow(
    tabela_cruzada, text_auto=True, aspect='auto',
    color_continuous_scale='Blues',
    title='Projetos por tema x região',
    labels=dict(color='Projetos'),
)
fig.update_layout(height=600)
fig.show()


Também vale acompanhar como os **temas mais financiados** evoluíram ao longo dos anos, um tema pode ser líder no total acumulado, mas ter perdido espaço recentemente (ou o contrário).


In [84]:
top_5_temas = por_tema.tail(5)['Tema'].tolist()

evolucao_temas = (
    df[df['Tema'].isin(top_5_temas)]
    .groupby(['Ano', 'Tema']).size()
    .reset_index(name='Projetos')
)

fig = px.line(
    evolucao_temas, x='Ano', y='Projetos', color='Tema',
    title='Evolução dos 5 temas mais financiados ao longo do tempo',
    markers=True,
)
fig.show()


Por fim, comparar o **valor total investido por região** (não só a contagem de projetos) mostra se a liderança em número de projetos também se traduz em liderança financeira.

In [85]:
fig = px.box(
    df, x='Região Geográfica', y='Valor Total do Projeto (R$)',
    title='Distribuição do valor dos projetos por região',
    log_y=True, color='Região Geográfica',
)
fig.update_layout(showlegend=False)
fig.show()


O boxplot está em **escala logarítmica** no eixo Y (`log_y=True`) necessário porque, como no histograma de valores, a distribuição é muito assimétrica: sem o log, os poucos projetos de valor altíssimo (como os outliers de R$ 4M+ no Sudeste) esmagariam visualmente todas as caixas na parte de baixo do gráfico.

Um ponto que chama atenção: o **Sudeste** tem o maior número de projetos (413) e concentra os maiores outliers (até R$ 4,05M), mas isso não significa que seus projetos sejam, em geral, os mais caros, a **mediana** do Sudeste (R$ 170,8 mil) é na verdade a **mais baixa entre as cinco regiões**. Quem tem a maior mediana é o **Nordeste** (R$ 230 mil), mesmo com menos projetos (312) e sem outliers tão extremos. Ou seja: liderança em quantidade de projetos e liderança em valor típico por projeto são coisas diferentes, o Sudeste lidera a primeira, o Nordeste lidera a segunda.

## 5. Conclusões

- **Educação Ambiental** é o tema mais financiado (319 projetos), à frente de Gestão de Áreas Protegidas e Conservação e Manejo da Biodiversidade.
- **Mata Atlântica** é o bioma com mais projetos (757, contando cada bioma individualmente em células com mais de um bioma), mais que Amazônia e Cerrado somados.
- O **Sudeste** lidera em número de projetos por região, o que pode refletir tanto maior capacidade institucional de captar recursos federais quanto maior densidade de instituições executoras na região, os dados não permitem distinguir essas hipóteses sem informação adicional.
- Houve um **pico de atividade entre 1993 e 2006**, um hiato quase completo entre 2007 e 2023, e uma retomada em 2024, um padrão que sugere mudança de política de financiamento, não queda de demanda.
- A distribuição de **valor por projeto é assimétrica**: a mediana (R$ 176 mil) é bem menor que a média (R$ 275 mil), indicando que uma minoria de projetos de grande porte concentra parte relevante dos recursos.
- A base tem **problemas de qualidade pontuais** (2 registros com duração negativa, inconsistências de espaçamento em campos categóricos) que precisam de tratamento antes de qualquer análise e que foram documentados e corrigidos.
